In [24]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\uniab\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\uniab\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [25]:
df = pd.read_csv("IMDB.csv")
df = df.sample(500)
df.to_csv("data,csv", index= False)
df.head(8)

,review,sentiment
238,I want very much to believe that the above quo...,negative
935,"The superb star quality of Gerard Philipe, who...",positive
308,"Definitely a very good idea,screenplay was jus...",positive
762,Early 80's creature feature concerns a long ab...,positive
218,"I've gotta say, I usually like horror movies t...",negative
989,David Bryce's comments nearby are exceptionall...,negative
856,Ann-Margret did the best job she has ever done...,positive
263,Basil Rathbone and Nigel Bruce as Sherlock Hol...,positive


### Data Pre-Processing

In [26]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [27]:
df

,review,sentiment
238,I want very much to believe that the above quo...,negative
935,"The superb star quality of Gerard Philipe, who...",positive
308,"Definitely a very good idea,screenplay was jus...",positive
762,Early 80's creature feature concerns a long ab...,positive
218,"I've gotta say, I usually like horror movies t...",negative
...,...,...
543,i was like watching it right and i was all lik...,positive
148,Principally it is the story of two men who wer...,positive
577,I think you would have to be from the USA to g...,positive
919,This movie lacks in everything. Except Bobby d...,negative


In [28]:
df = normalize_text(df)
df.head()

,review,sentiment
238,want much believe quote specifically english s...,negative
935,superb star quality gerard philipe died way yo...,positive
308,definitely good idea screenplay ok could bette...,positive
762,early s creature feature concern long abandone...,positive
218,gotta say usually like horror movie never seen...,negative


In [29]:
df["sentiment"].value_counts()

sentiment
negative    254
positive    246
Name: count, dtype: int64

In [30]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [32]:
df["sentiment"] = df["sentiment"].map({'positive': 1, "negative":0})
df.head()

,review,sentiment
238,want much believe quote specifically english s...,0
935,superb star quality gerard philipe died way yo...,1
308,definitely good idea screenplay ok could bette...,1
762,early s creature feature concern long abandone...,1
218,gotta say usually like horror movie never seen...,0


In [33]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [34]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [36]:
import dagshub 
mlflow.set_tracking_uri("https://dagshub.com/uniabhi5684/Sentiment-Analysis-MLOPS-Project.mlflow")
dagshub.init(repo_owner='uniabhi5684', repo_name='Sentiment-Analysis-MLOPS-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")

2026-01-20 19:14:59,761 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/uniabhi5684/Sentiment-Analysis-MLOPS-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "uniabhi5684/Sentiment-Analysis-MLOPS-Project"

2026-01-20 19:14:59,815 - INFO - Initialized MLflow to track repo "uniabhi5684/Sentiment-Analysis-MLOPS-Project"


Repository uniabhi5684/Sentiment-Analysis-MLOPS-Project initialized!

2026-01-20 19:14:59,824 - INFO - Repository uniabhi5684/Sentiment-Analysis-MLOPS-Project initialized!


<Experiment: artifact_location='mlflow-artifacts:/f9cb8be42f3e403088e9c47b0e85e0e1', creation_time=1767017825714, experiment_id='0', last_update_time=1767017825714, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [37]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-01-20 19:15:01,775 - INFO - Starting MLflow run...


2026-01-20 19:15:02,325 - INFO - Logging preprocessing parameters...
2026-01-20 19:15:03,749 - INFO - Initializing Logistic Regression model...
2026-01-20 19:15:03,752 - INFO - Fitting the model...
2026-01-20 19:15:03,852 - INFO - Model training complete.
2026-01-20 19:15:03,853 - INFO - Logging model parameters...
2026-01-20 19:15:04,386 - INFO - Making predictions...
2026-01-20 19:15:04,390 - INFO - Calculating evaluation metrics...
2026-01-20 19:15:04,444 - INFO - Logging evaluation metrics...
2026-01-20 19:15:06,100 - INFO - Saving and logging the model...
2026/01/20 19:15:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/20 19:15:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026-01-20 19:15:40,516 - INFO - Model training and logging completed in 38.19 seconds.
2026-01-20 19:15:40,519 - INFO - Accuracy:

🏃 View run bold-asp-403 at: https://dagshub.com/uniabhi5684/Sentiment-Analysis-MLOPS-Project.mlflow/#/experiments/0/runs/5f1460cc17964933a47976cf4d4df65c
🧪 View experiment at: https://dagshub.com/uniabhi5684/Sentiment-Analysis-MLOPS-Project.mlflow/#/experiments/0
